# CAM Summary & Spatial Bias

            聚合所有 `results/cam/**/summary.csv`，并从 heatmap 图中估计空间偏置：

            - `edge_mass`: CAM 激活是否偏边缘
            - `center_mass`: 是否偏中心
            - `corner_mass`: 是否偏四角
            - `cam_cx/cam_cy`: 热点质心


In [ ]:

from pathlib import Path
import json
import math
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

EXP_ROOT = Path("/data/zengqiang/experiments/ncfm_medmnist_ablation_20260519")

def require_exp_root():
    if not EXP_ROOT.exists():
        raise FileNotFoundError(
            f"EXP_ROOT not found: {EXP_ROOT}. "
            "Edit EXP_ROOT in the first code cell to your experiment directory."
        )

def ensure_report_dir(*parts):
    path = EXP_ROOT / "reports" / "cam" / Path(*parts)
    path.mkdir(parents=True, exist_ok=True)
    return path

def read_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def normalize_group(name):
    if name == "real_train":
        return "real_train"
    if name.startswith("ipc10_"):
        return name[len("ipc10_"):]
    return name

def resolve_result_path(value):
    p = Path(str(value))
    if p.exists():
        return p
    if str(value).startswith("/"):
        return p
    q = EXP_ROOT / value
    return q

def load_eval_metrics():
    require_exp_root()
    rows = []
    for path in sorted((EXP_ROOT / "runs").glob("*/ipc10/*/eval_metrics_best.json")):
        item = read_json(path)
        item["dataset"] = path.parents[2].name
        item["group"] = path.parent.name
        item["metrics_path"] = str(path)
        rows.append(item)
    if not rows:
        return pd.DataFrame()
    df = pd.DataFrame(rows)
    order = {"A_pure_ncfd_wopsi": 0, "B_minmax_ncfm_psi": 1, "C_code_default_enhanced": 2}
    df["_order"] = df["group"].map(order).fillna(99)
    return df.sort_values(["dataset", "_order"]).drop(columns=["_order"])

def load_cam_summaries():
    require_exp_root()
    rows = []
    for path in sorted((EXP_ROOT / "results" / "cam").glob("*/*/summary.csv")):
        dataset = path.parents[1].name
        group = normalize_group(path.parent.name)
        df = pd.read_csv(path)
        if df.empty:
            continue
        df["dataset"] = dataset
        df["group"] = group
        df["summary_path"] = str(path)
        rows.append(df)
    if not rows:
        return pd.DataFrame()
    out = pd.concat(rows, ignore_index=True)
    for col in ["index", "y_true", "y_pred", "confidence", "correct", "cam_entropy", "topk_activation_ratio"]:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")
    return out

def cam_group_summary(cam_df):
    if cam_df.empty:
        return cam_df
    grouped = (
        cam_df.groupby(["dataset", "group"], as_index=False)
        .agg(
            n=("index", "count"),
            cam_acc=("correct", "mean"),
            mean_confidence=("confidence", "mean"),
            mean_entropy=("cam_entropy", "mean"),
            mean_top10_mass=("topk_activation_ratio", "mean"),
            correct_entropy=("cam_entropy", lambda s: s[cam_df.loc[s.index, "correct"] == 1].mean()),
            wrong_entropy=("cam_entropy", lambda s: s[cam_df.loc[s.index, "correct"] == 0].mean()),
            correct_top10_mass=("topk_activation_ratio", lambda s: s[cam_df.loc[s.index, "correct"] == 1].mean()),
            wrong_top10_mass=("topk_activation_ratio", lambda s: s[cam_df.loc[s.index, "correct"] == 0].mean()),
        )
    )
    order = {"real_train": 0, "A_pure_ncfd_wopsi": 1, "B_minmax_ncfm_psi": 2, "C_code_default_enhanced": 3}
    grouped["_order"] = grouped["group"].map(order).fillna(99)
    return grouped.sort_values(["dataset", "_order"]).drop(columns=["_order"])


In [ ]:

cam_df = load_cam_summaries()
summary = cam_group_summary(cam_df)
display(summary)

report_dir = ensure_report_dir()
if not summary.empty:
    summary.to_csv(report_dir / "cam_summary_grouped.csv", index=False)
    cam_df.to_csv(report_dir / "cam_summary_all_rows.csv", index=False)


In [ ]:

def read_cam_array(path_value):
    path = resolve_result_path(path_value)
    if not path.exists():
        return None
    img = np.asarray(Image.open(path).convert("RGB")).astype(np.float32) / 255.0
    # heatmap is saved as reddish RGB, red channel preserves the CAM intensity
    cam = img[..., 0]
    total = cam.sum()
    if total <= 1e-12:
        return cam
    return cam

def spatial_stats(cam):
    h, w = cam.shape
    total = float(cam.sum()) + 1e-12
    edge = max(1, round(min(h, w) * 0.15))
    edge_mask = np.zeros((h, w), dtype=bool)
    edge_mask[:edge, :] = True
    edge_mask[-edge:, :] = True
    edge_mask[:, :edge] = True
    edge_mask[:, -edge:] = True
    center_mask = np.zeros((h, w), dtype=bool)
    y0, y1 = h // 4, h - h // 4
    x0, x1 = w // 4, w - w // 4
    center_mask[y0:y1, x0:x1] = True
    corner_mask = np.zeros((h, w), dtype=bool)
    corner = max(1, round(min(h, w) * 0.2))
    corner_mask[:corner, :corner] = True
    corner_mask[:corner, -corner:] = True
    corner_mask[-corner:, :corner] = True
    corner_mask[-corner:, -corner:] = True
    yy, xx = np.mgrid[0:h, 0:w]
    cx = float((cam * xx).sum() / total) / max(w - 1, 1)
    cy = float((cam * yy).sum() / total) / max(h - 1, 1)
    left = float(cam[:, : w // 2].sum()) / total
    top = float(cam[: h // 2, :].sum()) / total
    return {
        "edge_mass": float(cam[edge_mask].sum() / total),
        "center_mass": float(cam[center_mask].sum() / total),
        "corner_mass": float(cam[corner_mask].sum() / total),
        "cam_cx": cx,
        "cam_cy": cy,
        "left_mass": left,
        "right_mass": 1.0 - left,
        "top_mass": top,
        "bottom_mass": 1.0 - top,
    }

rows = []
for _, row in cam_df.iterrows():
    cam = read_cam_array(row["cam_path"])
    if cam is None:
        continue
    rows.append({**row.to_dict(), **spatial_stats(cam)})

spatial_df = pd.DataFrame(rows)
if spatial_df.empty:
    print("No CAM heatmaps found. Run this notebook on the lab server or set EXP_ROOT correctly.")
else:
    spatial_grouped = spatial_df.groupby(["dataset", "group"], as_index=False).agg(
        n=("index", "count"),
        edge_mass=("edge_mass", "mean"),
        center_mass=("center_mass", "mean"),
        corner_mass=("corner_mass", "mean"),
        cam_cx=("cam_cx", "mean"),
        cam_cy=("cam_cy", "mean"),
        left_mass=("left_mass", "mean"),
        top_mass=("top_mass", "mean"),
    )
    display(spatial_grouped)
    spatial_df.to_csv(report_dir / "cam_spatial_bias_all_rows.csv", index=False)
    spatial_grouped.to_csv(report_dir / "cam_spatial_bias_grouped.csv", index=False)


In [ ]:

if "spatial_grouped" in globals() and not spatial_grouped.empty:
    for metric in ["edge_mass", "center_mass", "corner_mass"]:
        pivot = spatial_grouped.pivot_table(index="dataset", columns="group", values=metric, aggfunc="first")
        display(pivot)
        ax = pivot.plot(kind="bar", figsize=(9, 4), rot=0)
        ax.set_title(metric)
        ax.grid(axis="y", alpha=0.25)
        plt.tight_layout()
        plt.show()


## 解读提示

            - `edge_mass` 高：模型可能依赖边框/背景伪特征。
            - `corner_mass` 高：可能存在固定角落或采样位置偏置。
            - `center_mass` 高不一定坏，医学主体常位于中心；需和 real-trained CAM 对比。
